# Drifting ICNN

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from typing import Tuple, List

### Hyperparameters

In [ ]:
METHODS = ["kernel", "ot_direct", "npf"]
TARGET = "four_gaussians"
NUM_ITERS = 3000
BATCH_SIZE = 512
LR = 3e-4
INNER_STEPS = 10
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_PATH = "results/icnn_drifting_comparison_q.png"

# NPF (Vesseron & Cuturi, 2024) ICNN parameterization replaces the plain
# ICNN drift field. Hidden widths follow the same tapered convention as
# the previous ICNN baseline.
NPF_HIDDEN_DIMS = [512, 256, 128, 64]

# NPF outer/inner low-rank PSD ranks. With outer_rank ≥ input_dim (=2 for
# this 2D toy) the closed-form Gaussian-OT init is captured exactly.
NPF_OUTER_RANK = 4
NPF_INNER_RANK = 1
# softplus everywhere has f''>0, so the cascade can carry curvature from
# the first inner step (ELU's u≥0 has f''=0 at the operating point).
NPF_ACTIVATION = "softplus"
NPF_SOFTPLUS_BETA = 1.0
NPF_INIT_EPS = 1e-2  # live-at-init magnitude; T(x)≈x to O(eps)
NPF_INNER_LR = 1e-2  # matches the original ICNN ablation inner_lr
NPF_GRAD_CLIP = 5.0

# Sinkhorn regularization used by the main NPF / direct-OT runs.
# The ablation section sweeps this value.
SINKHORN_REG = 0.05

# NPF init mode: "identity" or "gaussian".
# "gaussian" sets ∇ψ to the closed-form affine OT map between Gaussian
# approximations of the first generated batch and the first target batch.
NPF_INIT_MODE = "gaussian"
NPF_INIT_BLEND = 1.0  # use 0.5 for a safer but weaker Gaussian start


### NPF: Input Convex Network parameterization
Imported from `npf_drift.py`. This module exposes:
* `NPFInputConvexPotential` — convex $\psi_\omega(z)$ with PSD outer base + non-negative cascade.
* `NPFDriftField` — Adam inner loop with `init_mode={"identity","gaussian"}` and a user-supplied Sinkhorn target function.
* `gaussian_ot_affine_map(x, y) -> (A, b)` — closed-form Brenier map between Gaussian approximations.
* `barycentric_target_simple` — direct-domain Sinkhorn target used by these 2D toy experiments.
The kernel-based baseline `compute_V_kernel` is kept inline in the next cell because it is not part of the NPF module.

In [ ]:
# ---- NPF + kernel-baseline drift fields ----
# NPF classes/helpers come from npf_drift.py (shared with the MNIST and
# width/depth ablation notebooks). The kernel baseline stays here because
# it is not part of the NPF module.
from npf_drift import (
    NPFInputConvexPotential,
    NPFDriftField,
    gaussian_ot_affine_map,
    sample_mean_cov,
    psd_matrix_power,
    barycentric_target_simple,
    sinkhorn_simple,
    count_parameters,
    make_tapered_hidden_dims,
)


def compute_V_kernel(x_gen: torch.Tensor, y_pos: torch.Tensor,
                     tau_list: Tuple[float, ...] = (0.02, 0.05, 0.2)):
    """
    Compute the drifting field V(x) following Algorithm 2 / Eq. 11 of the paper.

    Returns V (detached), such that the training loss is:
        target = sg(x_gen + V)
        loss = ||x_gen - target||²

    The loss value equals ||V||², which DECREASES as q → p.
    """
    N, D = x_gen.shape
    M = y_pos.shape[0]
    x = x_gen.detach()
    y = y_pos.detach()

    # Targets: [negatives (=generated), positives (=data)]
    targets = torch.cat([x, y], dim=0)  # [N+M, D]

    # ── Pairwise ℓ₂ distances ──
    dist = torch.cdist(x, targets)  # [N, N+M]

    # ── Distance normalization (Appendix A.6) ──
    # Normalize so mean pairwise distance ≈ √D
    scale = dist.mean().clamp(min=1e-6)
    dist_normed = dist * (np.sqrt(D) / scale)

    # ── Self-masking ──
    diag_mask = torch.zeros(N, N + M, device=x.device)
    diag_mask[:, :N] = torch.eye(N, device=x.device)
    dist_normed = dist_normed + diag_mask * 100.0

    # ── Force accumulation (NO per-temperature normalization) ──
    V = torch.zeros_like(x)

    for tau in tau_list:
        logits = -dist_normed / tau

        # Double softmax (paper's Alg. 2: softmax over y, then over x)
        aff_row = torch.softmax(logits, dim=-1)
        aff_col = torch.softmax(logits, dim=-2)
        affinity = torch.sqrt((aff_row * aff_col).clamp(min=1e-6))

        aff_neg = affinity[:, :N]      # negative affinities
        aff_pos = affinity[:, N:]      # positive affinities
        sum_pos = aff_pos.sum(-1, keepdim=True)
        sum_neg = aff_neg.sum(-1, keepdim=True)

        # Drifting coefficients: attract by positives, repel by negatives
        r_coeff_neg = -aff_neg * sum_pos
        r_coeff_pos = aff_pos * sum_neg
        R_coeff = torch.cat([r_coeff_neg, r_coeff_pos], dim=1)  # [N, N+M]

        # Force = weighted combination of (target - x) vectors
        force_R = R_coeff @ targets - R_coeff.sum(-1, keepdim=True) * x
        V = V + force_R  # Raw force, NO normalization

    return V.detach()


### NPF-based V (Sinkhorn OT displacement) and direct OT baseline

In [ ]:
# Direct-domain Sinkhorn is reused both as a baseline and as the
# barycentric-target generator passed to NPFDriftField. Both are imported
# from npf_drift (sinkhorn_simple, barycentric_target_simple) — the
# wrappers below pin the regularization to a fixed value the way the
# previous ICNN notebook did.


def compute_V_ot_direct(x_gen: torch.Tensor, y_pos: torch.Tensor,
                        reg: float = SINKHORN_REG):
    """Direct OT displacement V(x) = T_Sinkhorn(x) - x."""
    x = x_gen.detach()
    y = y_pos.detach()
    y_target = barycentric_target_simple(x, y, reg=reg, num_iters=100)
    return (y_target - x).detach()


def make_npf_target_fn(reg: float):
    """Closure over reg for NPFDriftField.sinkhorn_target_fn."""
    return lambda x, y: barycentric_target_simple(x, y, reg=reg, num_iters=100)


def make_npf_drift(dim: int = 2, hidden_dims=None,
                   inner_steps: int = INNER_STEPS,
                   sinkhorn_reg: float = SINKHORN_REG,
                   inner_lr: float = NPF_INNER_LR,
                   init_mode: str = NPF_INIT_MODE,
                   init_blend: float = NPF_INIT_BLEND,
                   init_eps: float = NPF_INIT_EPS,
                   outer_rank: int = NPF_OUTER_RANK,
                   inner_rank: int = NPF_INNER_RANK,
                   activation: str = NPF_ACTIVATION,
                   softplus_beta: float = NPF_SOFTPLUS_BETA,
                   grad_clip: float = NPF_GRAD_CLIP) -> NPFDriftField:
    """Construct an NPFDriftField with this notebook's defaults.

    Equivalent to the original ICNNDriftField constructor in spirit —
    `init_mode='gaussian'` triggers a one-shot closed-form affine init
    on the first batch (see set_gaussian_init_from_samples).
    """
    if hidden_dims is None:
        hidden_dims = NPF_HIDDEN_DIMS
    return NPFDriftField(
        dim=dim,
        hidden_dims=hidden_dims,
        outer_rank=outer_rank,
        inner_rank=inner_rank,
        activation=activation,
        softplus_beta=softplus_beta,
        init_eps=init_eps,
        inner_steps=inner_steps,
        inner_lr=inner_lr,
        grad_clip=grad_clip,
        sinkhorn_reg=sinkhorn_reg,
        sinkhorn_iters=100,
        sinkhorn_target_fn=make_npf_target_fn(sinkhorn_reg),
        init_mode=init_mode,
        init_blend=init_blend,
    )


### Generator

In [ ]:
class ToyGenerator(nn.Module):
    """
    Small MLP: ε ∈ R² → x ∈ R².

    The initial output distribution must span the target range.
    If the initial generator output is concentrated (std << target range),
    the kernel V gives near-zero force (all data points equidistant).
    """
    def __init__(self, noise_dim=2, data_dim=2, hidden_dim=256, output_scale=2.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, data_dim),
        )
        # Initialize so output has std ≈ output_scale
        # We need to account for variance propagation through the MLP
        self._init_output_scale(output_scale)

    def _init_output_scale(self, target_std):
        """Initialize so the output std ≈ target_std."""
        with torch.no_grad():
            # Check current output scale
            test_input = torch.randn(1000, 2)
            test_output = self.net(test_input)
            current_std = test_output.std().item()
            if current_std > 1e-6:
                scale_factor = target_std / current_std
                # Scale only the last layer to avoid disrupting internal representations
                self.net[-1].weight.mul_(scale_factor)
                self.net[-1].bias.mul_(scale_factor)

    def forward(self, eps):
        return self.net(eps)

### Target Distributions

In [ ]:
def sample_bimodal(n, device):
    mix = torch.rand(n, device=device) < 0.5
    x = torch.randn(n, 2, device=device) * 0.4
    x[mix, 0] += 2.0
    x[~mix, 0] -= 2.0
    return x

def sample_ring(n, device):
    theta = 2 * np.pi * torch.rand(n, device=device)
    r = 2.0 + 0.2 * torch.randn(n, device=device)
    return torch.stack([r * torch.cos(theta), r * torch.sin(theta)], -1)

def sample_four_gaussians(n, device):
    c = torch.tensor([[-2, -2], [-2, 2], [2, -2], [2, 2]],
                      device=device, dtype=torch.float32)
    return c[torch.randint(0, 4, (n,), device=device)] + 0.3 * torch.randn(n, 2, device=device)

### Training Loop

In [ ]:
def train(method='kernel', target='bimodal', num_iters=3000, batch_size=512,
          lr=3e-4, inner_steps=10, seed=42,
          sinkhorn_reg=SINKHORN_REG, npf_hidden_dims=None,
          device='cuda' if torch.cuda.is_available() else 'cpu'):

    torch.manual_seed(seed)
    device = torch.device(device)
    samplers = dict(bimodal=sample_bimodal, ring=sample_ring,
                    four_gaussians=sample_four_gaussians)
    sample_target = samplers[target]

    if npf_hidden_dims is None:
        npf_hidden_dims = NPF_HIDDEN_DIMS

    gen = ToyGenerator(hidden_dim=256, output_scale=2.0).to(device)
    gen_optim = optim.Adam(gen.parameters(), lr=lr)

    with torch.no_grad():
        test_out = gen(torch.randn(1000, 2, device=device))
        print(f"  [{method}] Initial output: mean={test_out.mean(0).cpu().numpy()}, "
              f"std={test_out.std(0).cpu().numpy()}")

    npf_drift = None
    if method == 'npf':
        npf_drift = make_npf_drift(
            dim=2,
            hidden_dims=npf_hidden_dims,
            inner_steps=inner_steps,
            sinkhorn_reg=sinkhorn_reg,
        ).to(device)
        with torch.no_grad():
            tz = torch.randn(8, 2, device=device)
            err = (npf_drift.psi.gradient(tz, False) - tz).abs().max().item()
            print(f"  [pre-batch init check] hidden_dims={npf_hidden_dims} | "
                  f"max |T(z)-z| = {err:.2e}; init_mode={NPF_INIT_MODE}")

    losses, v_norms, snapshots = [], [], []

    for it in range(num_iters):
        eps = torch.randn(batch_size, 2, device=device)
        y_pos = sample_target(batch_size, device)
        x_gen = gen(eps)

        if method == 'kernel':
            V = compute_V_kernel(x_gen, y_pos, tau_list=(0.02, 0.05, 0.2))
        elif method == 'ot_direct':
            V = compute_V_ot_direct(x_gen, y_pos, reg=sinkhorn_reg)
        else:
            V = npf_drift.compute_V(x_gen, y_pos)

        # Drifting loss:
        # L = ||f(ε) - sg(f(ε) + V(f(ε)))||²
        target_pts = (x_gen.detach() + V).detach()  # frozen target
        loss = ((x_gen - target_pts) ** 2).mean()
        v_norm = (V ** 2).mean().item()

        gen_optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gen.parameters(), 1.0)
        gen_optim.step()

        losses.append(loss.item())
        v_norms.append(v_norm)

        if it % (num_iters // 8) == 0 or it == num_iters - 1:
            with torch.no_grad():
                snap_x = gen(torch.randn(1000, 2, device=device)).cpu().numpy()
            snapshots.append((it, snap_x))
            print(f"[{method}] iter {it:5d} | loss: {loss.item():.6f} | "
                  f"||V||²: {v_norm:.6f}")

    return dict(losses=losses, v_norms=v_norms, snapshots=snapshots,
                method=method, target=target,
                sinkhorn_reg=float(sinkhorn_reg),
                npf_hidden_dims=list(npf_hidden_dims) if method == 'npf' else None)


In [ ]:
results = []
for m in METHODS:
    print(f"\n{'=' * 60}\nTraining: {m}\n{'=' * 60}")
    results.append(train(
        method=m, target=TARGET, num_iters=NUM_ITERS,
        batch_size=BATCH_SIZE, lr=LR,
        inner_steps=INNER_STEPS, seed=SEED,
        sinkhorn_reg=SINKHORN_REG,
        npf_hidden_dims=NPF_HIDDEN_DIMS,
        device=DEVICE))


### Visualization

In [ ]:
def plot_results(results_list, target='bimodal', device='cpu',
                 save=False, save_path='icnn_drifting_comparison.png'):
    n_methods = len(results_list)
    n_snaps = len(results_list[0]['snapshots'])
    fig = plt.figure(figsize=(4 * n_snaps + 4, 4 * n_methods + 4))
    gs = GridSpec(n_methods + 1, n_snaps + 1, figure=fig, hspace=0.3, wspace=0.3)

    samplers = dict(bimodal=sample_bimodal, ring=sample_ring,
                    four_gaussians=sample_four_gaussians)
    y_gt = samplers[target](2000, torch.device('cpu')).numpy()

    for row, res in enumerate(results_list):
        for col, (it, snap) in enumerate(res['snapshots']):
            ax = fig.add_subplot(gs[row, col])
            ax.scatter(y_gt[:, 0], y_gt[:, 1], s=1, alpha=.3, c='blue', label='target $p$')
            ax.scatter(snap[:, 0], snap[:, 1], s=1, alpha=.3, c='orange', label='generated $q$')
            ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.set_aspect('equal')
            ax.set_title(f'iter {it}', fontsize=10)
            if col == 0:
                ax.set_ylabel(res['method'], fontsize=12, fontweight='bold')
            if row == 0 and col == 0:
                ax.legend(fontsize=7, loc='upper left')
        ax_l = fig.add_subplot(gs[row, -1])
        ax_l.semilogy(res['v_norms'], lw=.8)
        ax_l.set_xlabel('iter'); ax_l.set_ylabel('$\\|V\\|^2$'); ax_l.grid(True, alpha=.3)
        ax_l.set_title(f'{res["method"]}: $\\|V\\|^2$', fontsize=10)

    ax = fig.add_subplot(gs[-1, :])
    for i, res in enumerate(results_list):
        ax.semilogy(res['v_norms'], label=res['method'], lw=1.5, alpha=.8)
    ax.set_xlabel('iteration', fontsize=12)
    ax.set_ylabel('$\\|V\\|^2$ (log)', fontsize=12)
    ax.set_title('Convergence Comparison', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11); ax.grid(True, alpha=.3)

    if save:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved to {save_path}")

    plt.close(fig)
    return fig

In [ ]:
plot_results(results, TARGET, DEVICE)

## Gaussian metrics

Run the helper below after training to turn the current visual result into preliminary quantitative evidence.

In [ ]:

# Suggested metrics for four-Gaussian preliminary results.
# Run after `results = [...]` has been computed.

def four_gaussian_metrics(samples, radius=0.75):
    """Simple held-out diagnostics for the four-Gaussian target."""
    x = torch.as_tensor(samples, dtype=torch.float32)
    centers = torch.tensor([[-2., -2.], [-2., 2.], [2., -2.], [2., 2.]])
    d = torch.cdist(x, centers)
    nearest = d.argmin(dim=1)
    min_dist = d.min(dim=1).values

    counts = torch.bincount(nearest, minlength=4).float()
    mass = counts / counts.sum().clamp_min(1.0)
    close_mass = torch.stack([
        ((nearest == k) & (min_dist < radius)).float().mean()
        for k in range(4)
    ])
    captured = int((close_mass > 0.02).sum().item())
    off_manifold = float((min_dist >= radius).float().mean().item())

    return {
        "captured_modes": captured,
        "mass_per_mode": mass.tolist(),
        "close_mass_per_mode": close_mass.tolist(),
        "off_manifold_frac": off_manifold,
        "mean_nearest_center_dist": float(min_dist.mean().item()),
    }

for res in results:
    final_samples = res["snapshots"][-1][1]
    print(res["method"], four_gaussian_metrics(final_samples))


## ICNN initialization sanity check

This checks the two initializers before running long experiments. With `identity`, `∇ψ(x)` should be almost exactly `x`. With `gaussian`, `∇ψ(x)` should match the closed-form affine Gaussian OT map computed from one generated batch and one target batch.


In [ ]:
def check_npf_initializations(target=TARGET, batch_size=BATCH_SIZE,
                              device=DEVICE, seed=SEED):
    """Sanity-check the two NPF init modes before running long experiments.

    With `identity`, ∇ψ(x) should match x to O(NPF_INIT_EPS).
    With `gaussian`, ∇ψ(x) should match the closed-form affine OT map
    (A x + b) computed from one batch of generated samples and one batch
    of targets.
    """
    torch.manual_seed(seed)
    device = torch.device(device)
    sample_target = dict(
        bimodal=sample_bimodal,
        ring=sample_ring,
        four_gaussians=sample_four_gaussians,
    )[target]

    gen = ToyGenerator(hidden_dim=256, output_scale=2.0).to(device)
    x = gen(torch.randn(batch_size, 2, device=device)).detach()
    y = sample_target(batch_size, device).detach()

    # identity init
    psi_identity = NPFInputConvexPotential(
        input_dim=2, hidden_sizes=NPF_HIDDEN_DIMS,
        outer_rank=NPF_OUTER_RANK, inner_rank=NPF_INNER_RANK,
        activation=NPF_ACTIVATION, softplus_beta=NPF_SOFTPLUS_BETA,
        init_eps=NPF_INIT_EPS,
    ).to(device)
    psi_identity.init_as_identity()
    identity_err = (psi_identity.gradient(x, create_graph=False) - x).abs().max().item()

    # gaussian init
    psi_gaussian = NPFInputConvexPotential(
        input_dim=2, hidden_sizes=NPF_HIDDEN_DIMS,
        outer_rank=NPF_OUTER_RANK, inner_rank=NPF_INNER_RANK,
        activation=NPF_ACTIVATION, softplus_beta=NPF_SOFTPLUS_BETA,
        init_eps=NPF_INIT_EPS,
    ).to(device)
    A, b = psi_gaussian.set_gaussian_init_from_samples(x, y, blend=1.0)
    T_affine = x @ A.T + b
    T_npf = psi_gaussian.gradient(x, create_graph=False)
    gaussian_err = (T_npf - T_affine).abs().max().item()

    print(f"hidden dims: {NPF_HIDDEN_DIMS}")
    print(f"identity init: max |∇ψ(x)-x| = {identity_err:.3e}")
    print(f"gaussian init: max |∇ψ(x)-(Ax+b)| = {gaussian_err:.3e}")
    print(f"gaussian init drift norm ||Ax+b-x||² = "
          f"{((T_affine - x) ** 2).mean().item():.6f}")
    return {
        "identity_err": identity_err,
        "gaussian_err": gaussian_err,
        "gaussian_init_v_norm": ((T_affine - x) ** 2).mean().item(),
        "A": A.detach().cpu().numpy(),
        "b": b.detach().cpu().numpy(),
    }


# Backwards-compatible alias for any downstream code referring to the old name.
check_icnn_initializations = check_npf_initializations

# Run this once before the ablation.
init_check = check_npf_initializations()
init_check


## ICNN architecture + Sinkhorn regularization ablation

This section answers two questions:

1. **ICNN expressiveness:** what happens to the drift norm / loss as the ICNN becomes wider and deeper?
2. **Sinkhorn regularization:** how sensitive are the ICNN targets and final drift norm to `sinkhorn_reg`?

The architecture convention is now tapered:

```python
make_tapered_hidden_dims(width=512, depth=4) == [512, 256, 128, 64]
```

So this ablation does **not** test `[512, 512, 512, 512]`. It tests progressively larger tapered ICNNs.

Main quantities to report:

- `tail_mean_v_norm`: moving average of `||V||²` over the last 10% of training.
- `tail_mean_inner_loss`: whether `∇ψ(x)` is actually fitting the Sinkhorn barycentric target.
- `tail_mean_target_v_norm`: how strong the direct Sinkhorn target is for that regularization.
- `final_fit_ratio`: whether the ICNN drift is much smaller/larger than the target drift.
- `time_per_iter_sec`: cost of wider/deeper ICNNs.


In [ ]:
import os, time
import pandas as pd

# `make_tapered_hidden_dims` and `count_parameters` are imported from the
# npf_drift module (see the import block earlier in the notebook).


def train_npf_configuration(
    hidden_dims=None,
    width=None,
    depth=None,
    target="four_gaussians",
    num_iters=1000,
    batch_size=512,
    lr=3e-4,
    inner_steps=10,
    seed=42,
    device=None,
    sinkhorn_reg=0.05,
    inner_lr=NPF_INNER_LR,
    init_mode=NPF_INIT_MODE,
    init_blend=NPF_INIT_BLEND,
    init_eps=NPF_INIT_EPS,
    outer_rank=NPF_OUTER_RANK,
    inner_rank=NPF_INNER_RANK,
    activation=NPF_ACTIVATION,
    softplus_beta=NPF_SOFTPLUS_BETA,
    grad_clip=NPF_GRAD_CLIP,
    log_every=None,
):
    """Train only the NPF method for one architecture + one Sinkhorn reg.

    You can either pass `hidden_dims` directly or pass `(width, depth)`,
    in which case hidden dims are tapered: `make_tapered_hidden_dims(width, depth)`.

    Returns the per-iteration curves (`v_norms`, `inner_losses`,
    `target_v_norms`, `fit_ratios`) plus the tail-mean summary metrics.
    """
    if device is None:
        device = DEVICE
    device = torch.device(device)

    if hidden_dims is None:
        if width is None or depth is None:
            hidden_dims = NPF_HIDDEN_DIMS
        else:
            hidden_dims = make_tapered_hidden_dims(width=width, depth=depth)
    hidden_dims = list(map(int, hidden_dims))
    width = int(hidden_dims[0])
    depth = int(len(hidden_dims))
    hidden_dims_str = "-".join(map(str, hidden_dims))

    # Same generator initialization for all configs with the same seed.
    torch.manual_seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)

    samplers = dict(
        bimodal=sample_bimodal,
        ring=sample_ring,
        four_gaussians=sample_four_gaussians,
    )
    sample_target = samplers[target]

    gen = ToyGenerator(hidden_dim=256, output_scale=2.0).to(device)
    gen_optim = optim.Adam(gen.parameters(), lr=lr)

    npf_drift = NPFDriftField(
        dim=2,
        hidden_dims=hidden_dims,
        outer_rank=outer_rank,
        inner_rank=inner_rank,
        activation=activation,
        softplus_beta=softplus_beta,
        init_eps=init_eps,
        inner_steps=inner_steps,
        inner_lr=inner_lr,
        grad_clip=grad_clip,
        sinkhorn_reg=sinkhorn_reg,
        sinkhorn_iters=100,
        sinkhorn_target_fn=make_npf_target_fn(sinkhorn_reg),
        init_mode=init_mode,
        init_blend=init_blend,
    ).to(device)

    # Reset RNG after architecture-dependent NPF initialization so the
    # noise/data stream is comparable across width/depth/reg.
    torch.manual_seed(seed + 10_000)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed + 10_000)

    v_norms, losses, inner_losses, target_v_norms, fit_ratios = [], [], [], [], []
    gaussian_init_v_norm = None

    if log_every is None:
        log_every = max(1, num_iters // 5)

    t0 = time.time()
    for it in range(num_iters):
        eps = torch.randn(batch_size, 2, device=device)
        y_pos = sample_target(batch_size, device)
        x_gen = gen(eps)

        V, stats = npf_drift.compute_V_with_stats(x_gen, y_pos)

        target_pts = (x_gen.detach() + V).detach()
        loss = ((x_gen - target_pts) ** 2).mean()

        gen_optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gen.parameters(), 1.0)
        gen_optim.step()

        v_norm = (V ** 2).mean().item()
        losses.append(loss.item())
        v_norms.append(v_norm)
        inner_losses.append(stats["inner_loss"])
        target_v_norms.append(stats["target_v_norm"])
        fit_ratios.append(stats["fit_ratio"])
        if stats.get("gaussian_init_v_norm") is not None:
            gaussian_init_v_norm = stats["gaussian_init_v_norm"]

        if it % log_every == 0 or it == num_iters - 1:
            print(
                f"[dims={hidden_dims_str}, reg={sinkhorn_reg:g}, seed={seed}] "
                f"iter {it:5d}/{num_iters} | "
                f"||V||²={v_norm:.6f} | "
                f"inner={stats['inner_loss']:.6f} | "
                f"target={stats['target_v_norm']:.6f}"
            )

    elapsed = time.time() - t0
    tail = max(1, num_iters // 10)

    with torch.no_grad():
        final_samples = gen(torch.randn(2000, 2, device=device)).cpu().numpy()

    return {
        "width": int(width),
        "depth": int(depth),
        "hidden_dims": hidden_dims,
        "hidden_dims_str": hidden_dims_str,
        "seed": int(seed),
        "target": target,
        "num_iters": int(num_iters),
        "batch_size": int(batch_size),
        "inner_steps": int(inner_steps),
        "sinkhorn_reg": float(sinkhorn_reg),
        "inner_lr": float(inner_lr),
        "init_mode": init_mode,
        "init_blend": float(init_blend),
        "gaussian_init_v_norm": gaussian_init_v_norm,
        "param_count": int(count_parameters(npf_drift.psi)),
        "time_sec": float(elapsed),
        "time_per_iter_sec": float(elapsed / max(1, num_iters)),
        "losses": losses,
        "v_norms": v_norms,
        "inner_losses": inner_losses,
        "target_v_norms": target_v_norms,
        "fit_ratios": fit_ratios,
        "final_samples": final_samples,
        "final_v_norm": float(v_norms[-1]),
        "tail_mean_v_norm": float(np.mean(v_norms[-tail:])),
        "best_v_norm": float(np.min(v_norms)),
        "final_inner_loss": float(inner_losses[-1]),
        "tail_mean_inner_loss": float(np.mean(inner_losses[-tail:])),
        "final_target_v_norm": float(target_v_norms[-1]),
        "tail_mean_target_v_norm": float(np.mean(target_v_norms[-tail:])),
        "final_fit_ratio": float(fit_ratios[-1]),
    }


# Backwards-compatible aliases for any downstream code that still uses the
# previous ICNN-named entry points.
train_icnn_configuration = train_npf_configuration

def train_icnn_architecture(width=128, depth=3, **kwargs):
    return train_npf_configuration(width=width, depth=depth, **kwargs)


def run_npf_architecture_sinkhorn_ablation(
    widths=(128, 256, 512),
    depths=(2, 3, 4),
    sinkhorn_regs=(0.02, 0.05, 0.1),
    seeds=(42,),
    target="four_gaussians",
    num_iters=1000,
    batch_size=512,
    inner_steps=10,
    lr=3e-4,
    inner_lr=NPF_INNER_LR,
    init_mode=NPF_INIT_MODE,
    init_blend=NPF_INIT_BLEND,
    init_eps=NPF_INIT_EPS,
    outer_rank=NPF_OUTER_RANK,
    activation=NPF_ACTIVATION,
    softplus_beta=NPF_SOFTPLUS_BETA,
    device=None,
    save_dir="results/icnn_architecture_sinkhorn_ablation",
):
    """Joint ablation over tapered NPF architecture and Sinkhorn reg.

    Quick first pass:
        widths=(256, 512), depths=(3, 4), sinkhorn_regs=(0.02, 0.05, 0.1),
        seeds=(42,), num_iters=600

    Report-quality:
        widths=(128, 256, 512), depths=(2, 3, 4, 5),
        sinkhorn_regs=(0.01, 0.02, 0.05, 0.1, 0.2),
        seeds=(0, 1, 2, 3, 4), num_iters=3000
    """
    os.makedirs(save_dir, exist_ok=True)
    records, runs = [], []

    total = len(widths) * len(depths) * len(sinkhorn_regs) * len(seeds)
    job = 0

    for reg in sinkhorn_regs:
        for depth in depths:
            for width in widths:
                hidden_dims = make_tapered_hidden_dims(width=width, depth=depth)
                for seed in seeds:
                    job += 1
                    print("\n" + "=" * 90)
                    print(f"Ablation job {job}/{total}: hidden_dims={hidden_dims}, "
                          f"sinkhorn_reg={reg}, seed={seed}")
                    print("=" * 90)

                    run = train_npf_configuration(
                        hidden_dims=hidden_dims,
                        target=target,
                        num_iters=num_iters,
                        batch_size=batch_size,
                        lr=lr,
                        inner_steps=inner_steps,
                        seed=seed,
                        device=device,
                        sinkhorn_reg=reg,
                        inner_lr=inner_lr,
                        init_mode=init_mode,
                        init_blend=init_blend,
                        init_eps=init_eps,
                        outer_rank=outer_rank,
                        activation=activation,
                        softplus_beta=softplus_beta,
                    )

                    runs.append(run)
                    records.append({
                        "width": run["width"],
                        "depth": run["depth"],
                        "hidden_dims_str": run["hidden_dims_str"],
                        "seed": run["seed"],
                        "sinkhorn_reg": run["sinkhorn_reg"],
                        "param_count": run["param_count"],
                        "final_v_norm": run["final_v_norm"],
                        "tail_mean_v_norm": run["tail_mean_v_norm"],
                        "best_v_norm": run["best_v_norm"],
                        "final_inner_loss": run["final_inner_loss"],
                        "tail_mean_inner_loss": run["tail_mean_inner_loss"],
                        "final_target_v_norm": run["final_target_v_norm"],
                        "tail_mean_target_v_norm": run["tail_mean_target_v_norm"],
                        "final_fit_ratio": run["final_fit_ratio"],
                        "time_sec": run["time_sec"],
                        "time_per_iter_sec": run["time_per_iter_sec"],
                        "num_iters": run["num_iters"],
                        "batch_size": run["batch_size"],
                        "inner_steps": run["inner_steps"],
                        "inner_lr": run["inner_lr"],
                        "init_mode": run["init_mode"],
                        "init_blend": run["init_blend"],
                        "gaussian_init_v_norm": run["gaussian_init_v_norm"],
                    })

                    df = pd.DataFrame(records)
                    df.to_csv(os.path.join(save_dir, "ablation_summary_partial.csv"), index=False)

    df = pd.DataFrame(records)
    df.to_csv(os.path.join(save_dir, "ablation_summary.csv"), index=False)
    return df, runs


# Backwards-compatible aliases.
run_icnn_architecture_sinkhorn_ablation = run_npf_architecture_sinkhorn_ablation


def run_width_depth_ablation(widths=(128, 256, 512), depths=(2, 3, 4), seeds=(42,), **kwargs):
    return run_npf_architecture_sinkhorn_ablation(
        widths=widths, depths=depths, seeds=seeds,
        sinkhorn_regs=(kwargs.pop("sinkhorn_reg", SINKHORN_REG),),
        **kwargs,
    )


def summarize_ablation(df):
    """Aggregate across seeds. Use `tail_mean_v_norm_mean` as the main number."""
    metrics = [
        "param_count",
        "final_v_norm",
        "tail_mean_v_norm",
        "best_v_norm",
        "final_inner_loss",
        "tail_mean_inner_loss",
        "final_target_v_norm",
        "tail_mean_target_v_norm",
        "final_fit_ratio",
        "time_per_iter_sec",
        "gaussian_init_v_norm",
    ]

    grouped = (
        df.groupby(["sinkhorn_reg", "depth", "width", "hidden_dims_str"])[metrics]
        .agg(["mean", "std"])
        .reset_index()
    )
    grouped.columns = [
        "_".join([str(x) for x in col if x != ""]).strip("_")
        for col in grouped.columns.to_flat_index()
    ]
    grouped = grouped.sort_values("tail_mean_v_norm_mean")
    return grouped


def plot_architecture_heatmap(df, metric="tail_mean_v_norm",
                              sinkhorn_reg=None, save_path=None):
    """Heatmap rows=depth, cols=width. Cell values averaged over seeds."""
    if sinkhorn_reg is not None:
        plot_df = df[np.isclose(df["sinkhorn_reg"], sinkhorn_reg)].copy()
        title_suffix = f" | sinkhorn_reg={sinkhorn_reg:g}"
    else:
        plot_df = df.copy()
        title_suffix = " | averaged over sinkhorn_reg"

    pivot = plot_df.groupby(["depth", "width"])[metric].mean().unstack("width")
    depths = list(pivot.index)
    widths = list(pivot.columns)

    fig, ax = plt.subplots(figsize=(1.4 * len(widths) + 3, 1.0 * len(depths) + 2))
    im = ax.imshow(pivot.values, aspect="auto")
    ax.set_xticks(range(len(widths)))
    ax.set_xticklabels(widths)
    ax.set_yticks(range(len(depths)))
    ax.set_yticklabels(depths)
    ax.set_xlabel("NPF starting width, tapered by /2 each layer")
    ax.set_ylabel("NPF depth")
    ax.set_title(metric + title_suffix)

    for i in range(len(depths)):
        for j in range(len(widths)):
            val = pivot.values[i, j]
            ax.text(j, i, f"{val:.3g}", ha="center", va="center")

    fig.colorbar(im, ax=ax, label=metric)
    fig.tight_layout()
    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    return fig, ax


def plot_sinkhorn_reg_ablation(df, metric="tail_mean_v_norm", save_path=None):
    """Plot metric vs sinkhorn_reg for each tapered NPF architecture."""
    mean_df = (
        df.groupby(["hidden_dims_str", "sinkhorn_reg"])[metric]
        .mean()
        .reset_index()
        .sort_values("sinkhorn_reg")
    )

    fig, ax = plt.subplots(figsize=(8, 5))
    for hidden_dims_str, sub in mean_df.groupby("hidden_dims_str"):
        ax.plot(sub["sinkhorn_reg"], sub[metric], marker="o", label=hidden_dims_str)

    ax.set_xscale("log")
    ax.set_xlabel("Sinkhorn regularization")
    ax.set_ylabel(metric)
    ax.set_title(f"Sinkhorn regularization ablation: {metric}")
    ax.grid(True, alpha=0.3)
    if mean_df["hidden_dims_str"].nunique() <= 10:
        ax.legend(title="hidden dims", fontsize=8)
    fig.tight_layout()
    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    return fig, ax


def plot_ablation_curves(runs, metric="v_norms", save_path=None, max_curves=15):
    """Plot training curves; only first `max_curves` for readability."""
    fig, ax = plt.subplots(figsize=(8, 5))
    for run in runs[:max_curves]:
        label = f"dims={run['hidden_dims_str']}, reg={run['sinkhorn_reg']:g}, seed={run['seed']}"
        y = run[metric]
        ax.semilogy(y, linewidth=1.0, alpha=0.85, label=label)

    ax.set_xlabel("iteration")
    ylabel = {
        "v_norms": r"$\|V\|^2$ / generator loss",
        "inner_losses": r"NPF inner loss $\|\nabla\psi(x)-\bar y\|^2$",
        "target_v_norms": r"direct OT target drift $\|\bar y-x\|^2$",
        "fit_ratios": r"$\|V_{NPF}\|^2 / \|\bar y-x\|^2$",
    }.get(metric, metric)
    ax.set_ylabel(ylabel)
    ax.set_title(f"NPF ablation curves: {metric}")
    ax.grid(True, alpha=0.3)
    if min(len(runs), max_curves) <= 12:
        ax.legend(fontsize=8)
    fig.tight_layout()
    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    return fig, ax


### Run the architecture + Sinkhorn ablation

Start with the quick grid below. It is intentionally small so you can verify that the code runs.

Once it works, switch to the report-quality grid in the comments.

Interpretation:

- If larger tapered ICNNs reduce both `tail_mean_v_norm` and `tail_mean_inner_loss`, ICNN capacity was limiting.
- If `tail_mean_inner_loss` falls but `tail_mean_v_norm` does not, the generator update or target regularization may be limiting.
- If very small `sinkhorn_reg` produces unstable or noisy curves, use this to justify a moderate regularization value.
- If large `sinkhorn_reg` gives overly smooth targets, expect lower/noisier mode specificity.


In [ ]:
# Quick first pass: enough to check trends without spending too much compute.
# The config width=512, depth=4 corresponds to [512, 256, 128, 64].
ABLATION_WIDTHS = [128, 256, 512]
ABLATION_DEPTHS = [2, 3, 4]
ABLATION_SINKHORN_REGS = [0.02, 0.05, 0.1]
ABLATION_SEEDS = [42]
ABLATION_ITERS = 1000

# Report-quality version:
# ABLATION_WIDTHS = [128, 256, 512]
# ABLATION_DEPTHS = [2, 3, 4, 5]
# ABLATION_SINKHORN_REGS = [0.01, 0.02, 0.05, 0.1, 0.2]
# ABLATION_SEEDS = [0, 1, 2, 3, 4]
# ABLATION_ITERS = 3000

ablation_df, ablation_runs = run_npf_architecture_sinkhorn_ablation(
    widths=ABLATION_WIDTHS,
    depths=ABLATION_DEPTHS,
    sinkhorn_regs=ABLATION_SINKHORN_REGS,
    seeds=ABLATION_SEEDS,
    target=TARGET,
    num_iters=ABLATION_ITERS,
    batch_size=BATCH_SIZE,
    inner_steps=INNER_STEPS,
    lr=LR,
    inner_lr=NPF_INNER_LR,
    init_mode=NPF_INIT_MODE,
    init_blend=NPF_INIT_BLEND,
    init_eps=NPF_INIT_EPS,
    outer_rank=NPF_OUTER_RANK,
    activation=NPF_ACTIVATION,
    softplus_beta=NPF_SOFTPLUS_BETA,
    device=DEVICE,
)

ablation_df


In [ ]:
ablation_summary = summarize_ablation(ablation_df)
ablation_summary


In [ ]:
for reg in ABLATION_SINKHORN_REGS:
    plot_architecture_heatmap(
        ablation_df,
        metric="tail_mean_v_norm",
        sinkhorn_reg=reg,
        save_path=f"results/icnn_architecture_sinkhorn_ablation/heatmap_tail_mean_v_norm_reg_{reg:g}.png",
    )

    plot_architecture_heatmap(
        ablation_df,
        metric="tail_mean_inner_loss",
        sinkhorn_reg=reg,
        save_path=f"results/icnn_architecture_sinkhorn_ablation/heatmap_tail_mean_inner_loss_reg_{reg:g}.png",
    )

plot_sinkhorn_reg_ablation(
    ablation_df,
    metric="tail_mean_v_norm",
    save_path="results/icnn_architecture_sinkhorn_ablation/sinkhorn_reg_tail_mean_v_norm.png",
)

plot_sinkhorn_reg_ablation(
    ablation_df,
    metric="tail_mean_inner_loss",
    save_path="results/icnn_architecture_sinkhorn_ablation/sinkhorn_reg_tail_mean_inner_loss.png",
)

plot_ablation_curves(
    ablation_runs,
    metric="v_norms",
    save_path="results/icnn_architecture_sinkhorn_ablation/curves_v_norms.png",
)

plot_ablation_curves(
    ablation_runs,
    metric="inner_losses",
    save_path="results/icnn_architecture_sinkhorn_ablation/curves_inner_losses.png",
)
